In [2]:
import numpy as np
import tkinter as tk
from tkinter import messagebox
import requests

GRID_SIZE = 4

ACTIONS = ["UP","DOWN","LEFT","RIGHT"]

alpha = 0.1
gamma = 0.9
epsilon = 0.15
episodes = 1000

TILE_SIZE_CM = 50   # default tile size

grid = np.zeros((GRID_SIZE,GRID_SIZE))

start=None
goal=None
obstacles=[]

mode="obstacle"

Q=np.zeros((GRID_SIZE,GRID_SIZE,len(ACTIONS)))

buttons={}
tile_entry=None

# ----------------------

def manhattan(a,b):
    return abs(a[0]-b[0])+abs(a[1]-b[1])

# ----------------------

def step(state,action):

    r,c=state

    moves={
        0:(-1,0),
        1:(1,0),
        2:(0,-1),
        3:(0,1)
    }

    dr,dc=moves[action]

    nr=r+dr
    nc=c+dc

    if nr<0 or nr>=GRID_SIZE or nc<0 or nc>=GRID_SIZE:
        return state,-5,False

    new_state=(nr,nc)

    if new_state in obstacles:
        return new_state,-100,True

    if new_state==goal:
        return new_state,100,True

    reward=-1

    old_dist=manhattan(state,goal)
    new_dist=manhattan(new_state,goal)

    if new_dist<old_dist:
        reward+=3
    else:
        reward-=3

    return new_state,reward,False

# ----------------------

def train():

    global Q,TILE_SIZE_CM

    if start is None or goal is None:
        messagebox.showerror("Error","Set Start and Goal first")
        return

    TILE_SIZE_CM = float(tile_entry.get())

    for ep in range(episodes):

        state=start
        visited=set()

        for step_count in range(50):

            if np.random.rand()<epsilon:
                action=np.random.randint(4)
            else:
                action=np.argmax(Q[state[0],state[1]])

            new_state,reward,done=step(state,action)

            if new_state in visited:
                reward-=15
            else:
                visited.add(new_state)

            old=Q[state[0],state[1],action]
            next_max=np.max(Q[new_state[0],new_state[1]])

            Q[state[0],state[1],action]=old+alpha*(reward+gamma*next_max-old)

            state=new_state

            if done:
                break

    print("\nQ TABLE\n",Q)

    policy=extract_policy()

    simulate_path(policy)

    send_policy(policy)

# ----------------------

def extract_policy():

    policy={}

    print("\nPOLICY")

    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):

            state=(r,c)

            if state in obstacles:
                continue

            if state==goal:
                policy[str(state)]="GOAL"
                print(state,"-> GOAL")
                continue

            action=np.argmax(Q[r,c])

            policy[str(state)]=ACTIONS[action]

            print(state,"->",ACTIONS[action])

    return policy

# ----------------------

def simulate_path(policy):

    global TILE_SIZE_CM

    state=start
    path=[state]

    steps=0

    for _ in range(20):

        if state==goal:
            break

        action=policy[str(state)]

        r,c=state

        if action=="UP":
            r-=1
        elif action=="DOWN":
            r+=1
        elif action=="LEFT":
            c-=1
        elif action=="RIGHT":
            c+=1

        state=(r,c)

        path.append(state)

        steps+=1

    distance = steps * TILE_SIZE_CM

    print("\nPATH:",path)
    print("Steps:",steps)
    print("Tile Size:",TILE_SIZE_CM,"cm")
    print("Total Distance:",distance,"cm")

# ----------------------

def send_policy(policy):

    state=start
    actions=[]

    for _ in range(20):

        if state==goal:
            actions.append("GOAL")
            break

        action=policy[str(state)]
        actions.append(action)

        r,c=state

        if action=="UP":
            r-=1
        elif action=="DOWN":
            r+=1
        elif action=="LEFT":
            c-=1
        elif action=="RIGHT":
            c+=1

        state=(r,c)

    data={
        "tile_size":TILE_SIZE_CM,
        "path":actions
    }

    url="http://192.168.4.1/policy"

    try:
        requests.post(url,json=data)
        print("Path sent to ESP32:",actions)
    except:
        print("ESP32 not connected")

# ----------------------

def cell_click(r,c):

    global start,goal

    if mode=="start":

        if start:
            buttons[start].config(bg="white",text="")

        start=(r,c)
        buttons[(r,c)].config(bg="blue",text="S")

    elif mode=="goal":

        if goal:
            buttons[goal].config(bg="white",text="")

        goal=(r,c)
        buttons[(r,c)].config(bg="green",text="G")

    elif mode=="obstacle":

        if (r,c) not in obstacles:
            obstacles.append((r,c))

        buttons[(r,c)].config(bg="black",text="X")

    elif mode=="erase":

        if (r,c) in obstacles:
            obstacles.remove((r,c))

        if start==(r,c):
            start=None

        if goal==(r,c):
            goal=None

        buttons[(r,c)].config(bg="white",text="")

# ----------------------

def set_mode(m):

    global mode
    mode=m

# ----------------------

def build_gui():

    global tile_entry

    root=tk.Tk()
    root.title("RL Robot Trainer")

    control=tk.Frame(root)
    control.pack()

    tk.Button(control,text="Set Start",command=lambda:set_mode("start")).grid(row=0,column=0)
    tk.Button(control,text="Set Goal",command=lambda:set_mode("goal")).grid(row=0,column=1)
    tk.Button(control,text="Add Obstacle",command=lambda:set_mode("obstacle")).grid(row=0,column=2)
    tk.Button(control,text="Erase",command=lambda:set_mode("erase")).grid(row=0,column=3)
    tk.Button(control,text="Train",command=train).grid(row=0,column=4)

    tk.Label(control,text="Tile Size (cm)").grid(row=1,column=0)

    tile_entry=tk.Entry(control,width=10)
    tile_entry.insert(0,"40")
    tile_entry.grid(row=1,column=1)

    grid_frame=tk.Frame(root)
    grid_frame.pack()

    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):

            b=tk.Button(grid_frame,width=8,height=4,bg="white",
                        command=lambda r=r,c=c:cell_click(r,c))

            b.grid(row=r,column=c)

            buttons[(r,c)]=b

    root.mainloop()

build_gui()


Q TABLE
 [[[ 1.39615605e+00  7.34042111e+01  7.27514953e+00  1.31507873e+01]
  [ 1.77122465e+01  1.34927165e+01 -2.09332250e+00  7.80147850e+01]
  [-2.00000000e+00  2.39676940e+01  3.73247716e+00  9.14631303e+01]
  [-2.00000000e+00  9.99303801e+01  3.32676381e+00  3.40669117e+01]]

 [[ 5.64334195e+01  5.06459899e+01  6.61292823e+01  8.36179714e+01]
  [ 5.31835190e+01  6.41439683e+01  7.04939540e+01  9.16465056e+01]
  [ 6.92852958e+01  7.73086128e+01  6.25526995e+01  1.00000000e+02]
  [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]]

 [[ 1.35623607e+01 -2.00498800e+00 -3.54007605e+00  7.21744690e+01]
  [ 2.45346044e+01 -1.50516297e+00  1.98010887e+00  8.41796416e+01]
  [ 3.54603056e+01  6.22881281e-02  4.92115098e+00  9.19876447e+01]
  [ 9.99995570e+01  7.66497394e+00  6.27113250e+00  3.98545263e+01]]

 [[-2.57080000e+00 -3.98000000e+00 -3.80000000e+00 -1.69691074e+00]
  [-1.30000000e+00 -3.87110000e+00 -3.67291900e+00  1.49588208e+00]
  [-1.30000000e+00 -2.00000000e+